In [ ]:
import sys
import torch
from time import time
# Import custom utilities
sys.path.append('../utilities/')
# Import custom utilities
from utils_baseline import BaseUtilsImdb

In [ ]:
torch.manual_seed(42)
batch_size = 32
epochs = 4
lr=5e-5
shuffle = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
base_utils_obj = BaseUtilsImdb()

In [ ]:
tokenized_train, tokenized_val, tokenized_test = base_utils_obj.get_tokenized_datasets()

In [ ]:
results_path = './results/batch_size_{}_epochs_{}_lr_{}'.format(batch_size, epochs, lr)
log_path = './logs/batch_size_{}_epochs_{}_lr_{}'.format(batch_size, epochs, lr)

In [ ]:
from transformers import TrainingArguments, Trainer,DistilBertForSequenceClassification

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2,device_map=device)

In [ ]:
training_args=TrainingArguments(
    output_dir=results_path,
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=lr,
    logging_dir=log_path,
    logging_steps=10,
    eval_strategy='steps',
    save_steps=10,
    eval_steps=10,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='tensorboard',
    seed=42,
    run_name='Batch size: {}, Epochs: {}, LR: {}'.format(batch_size, epochs, lr),
    lr_scheduler_type='constant',
    warmup_steps=0,
)

In [ ]:
start_time = time()

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=base_utils_obj.tokenizer,
    compute_metrics=base_utils_obj.compute_metric_accuracy
)

In [ ]:
trainer.train()

In [ ]:
# Save the best model
'''
Keep a local copy of the best model
'''
best_model_path = "./best_model_{batch_size}_{epochs}_{lr}".format(batch_size=batch_size, epochs=epochs, lr=lr)
trainer.model.save_pretrained(best_model_path)

In [ ]:
base_utils_obj.write_time(start_time,time(),batch_size,epochs)

In [ ]:
predictions = trainer.predict(tokenized_test)
accuracy = predictions[-1]['test_accuracy']

In [ ]:
with open('./test_acc.txt','a+') as f:
    f.write('Batch size: {}, Epochs: {}, LR: {}, Accuracy: {}\n'.format(batch_size, epochs, lr, accuracy))